# Phase 6e Wave 2: Higher-Curvature Structure — Technical Notebook

Companion to `papers/paper40_higher_curvature/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/HigherCurvatureStructure.lean` (11 substantive theorems, 0 sorry, 0 new axioms; verified `propext, Classical.choice, Quot.sound` only via `lean_verify`).

**Structure (mirrors paper §2–§5):**
1. Setup — Wave 1 Christensen-Duff a_4 coefficients (input)
2. Curvature basis: Gauss-Bonnet density + Weyl-squared
3. Conformal-flatness biconditional + algebraic identity
4. Stelle-basis coefficients (α, β, γ) — closed forms + sign theorems
5. Main basis-change identity (cross-bridge to Wave 1)
6. Observational ceilings (LIGO / SRG / pulsar / Cassini)
7. Correctness-push: predictions vs ceilings
8. Falsifier: predictions strictly non-zero
9. Tracked Prop H_HigherCurvatureWithinObservationalBounds
10. Figure: predictions vs ceilings
11. Lean theorem inventory

All numerical content imports from `src.higher_curvature` — no inline physics redefinition.


## 1. Setup — Wave 1 Christensen-Duff a_4 coefficients

In [1]:
import numpy as np
import math
from src.core.constants import HEAT_KERNEL_PARAMS, HIGHER_CURVATURE_PARAMS
from src.core.formulas import (
    higher_curvature_R_sq_coefficient,
    higher_curvature_Ricci_sq_coefficient,
    higher_curvature_Riemann_sq_coefficient,
)

print(f"FOUR_PI_SQ = (4π)² = {HEAT_KERNEL_PARAMS['FOUR_PI_SQ']:.6f}")
print(f"Christensen-Duff Dirac a_4 rationals (per N_f, per (4π)²):")
print(f"  c_R²       = {HIGHER_CURVATURE_PARAMS['A4_R_SQ_PER_NF']:+.6f}  (= -5/(12·180) = -1/432)")
print(f"  c_Ricci²   = {HIGHER_CURVATURE_PARAMS['A4_RICCI_SQ_PER_NF']:+.6f}  (= +7/(12·180) = +7/2160)")
print(f"  c_Riemann² = {HIGHER_CURVATURE_PARAMS['A4_RIEMANN_SQ_PER_NF']:+.6f}  (= -12/(12·180) = -1/180)")

FOUR_PI_SQ = (4π)² = 157.913670
Christensen-Duff Dirac a_4 rationals (per N_f, per (4π)²):
  c_R²       = -0.002315  (= -5/(12·180) = -1/432)
  c_Ricci²   = +0.003241  (= +7/(12·180) = +7/2160)
  c_Riemann² = -0.005556  (= -12/(12·180) = -1/180)


## 2. Curvature basis: Gauss-Bonnet density + Weyl-squared

**Gauss-Bonnet 4D density:** $\mathcal{G} = R^2 - 4 R_{\mu\nu} R^{\mu\nu} + R_{\mu\nu\rho\sigma} R^{\mu\nu\rho\sigma}$

**Weyl-squared (trace decomposition in 4D):** $C^2 = R_{\mu\nu\rho\sigma}^2 - 2 R_{\mu\nu}^2 + \tfrac{1}{3} R^2$

**Lean defs:** `gaussBonnet4D`, `weylSquared4D`.

In [2]:
from src.higher_curvature import gauss_bonnet_4D, weyl_squared_4D

# Sanity at de Sitter (H=1): R²=144, R_μν²=36, R_μνρσ²=24
R_dS, Ric_dS, Riem_dS = 144.0, 36.0, 24.0
print(f"de Sitter (H=1): GB = {gauss_bonnet_4D(R_dS, Ric_dS, Riem_dS):.4f} (expect 24, χ_S^4 = 2)")
print(f"de Sitter (H=1): C² = {weyl_squared_4D(R_dS, Ric_dS, Riem_dS):+.4f} (expect 0; conformally flat)")

# Sanity at Schwarzschild vacuum (M=1, r=2): R=R_μν=0, R_μνρσ²=48 M²/r⁶
Riem_Sch = 48.0 / (2.0 ** 6)
print(f"Schwarzschild vac (M=1, r=2): Riem² = {Riem_Sch:.6f}")
print(f"  GB(0,0,Riem²) = {gauss_bonnet_4D(0,0,Riem_Sch):.6f}  (= Riem²; vacuum Euler density)")
print(f"  C²(0,0,Riem²) = {weyl_squared_4D(0,0,Riem_Sch):.6f}  (= Riem²; vacuum is pure Weyl)")

de Sitter (H=1): GB = 24.0000 (expect 24, χ_S^4 = 2)
de Sitter (H=1): C² = +0.0000 (expect 0; conformally flat)
Schwarzschild vac (M=1, r=2): Riem² = 0.750000
  GB(0,0,Riem²) = 0.750000  (= Riem²; vacuum Euler density)
  C²(0,0,Riem²) = 0.750000  (= Riem²; vacuum is pure Weyl)


## 3. Conformal-flatness biconditional + algebraic identity

**Lean theorem `weylSquared4D_eq_zero_iff_conformally_flat`:** $C^2 = 0 \iff R_{\mu\nu\rho\sigma}^2 = 2 R_{\mu\nu}^2 - \tfrac{1}{3} R^2$.

**Lean theorem `gaussBonnet_minus_weyl_eq_R_minus_Ricci_combination`:** $\mathcal{G} - C^2 = \tfrac{2}{3} R^2 - 2 R_{\mu\nu}^2$.

This is the algebraic engine for the basis change.

In [3]:
from src.higher_curvature import gauss_bonnet_combination_check, weyl_squared_de_sitter_zero, weyl_squared_schwarzschild_vacuum

print("Conformal-flatness check at de Sitter (multiple H values):")
for H in (0.5, 1.0, 2.0, 10.0):
    print(f"  H = {H}: weyl_squared_de_sitter_zero = {weyl_squared_de_sitter_zero(H)}")

print()
print("Schwarzschild vacuum (Weyl² = Riem²):")
for M, r in [(1.0, 2.0), (5.0, 10.0), (0.5, 1.5)]:
    print(f"  M={M}, r={r}: weyl_squared_schwarzschild_vacuum = {weyl_squared_schwarzschild_vacuum(M, r)}")

print()
print("GB - C² = (2/3) R² - 2 Ricci² identity (random inputs):")
import random
random.seed(101)
for i in range(5):
    R, Ric, Riem = (random.uniform(-50, 50) for _ in range(3))
    ok = gauss_bonnet_combination_check(R, Ric, Riem)
    print(f"  (R²={R:+6.2f}, Ric²={Ric:+6.2f}, Riem²={Riem:+6.2f}): identity holds = {ok}")

Conformal-flatness check at de Sitter (multiple H values):
  H = 0.5: weyl_squared_de_sitter_zero = True
  H = 1.0: weyl_squared_de_sitter_zero = True
  H = 2.0: weyl_squared_de_sitter_zero = True
  H = 10.0: weyl_squared_de_sitter_zero = True

Schwarzschild vacuum (Weyl² = Riem²):
  M=1.0, r=2.0: weyl_squared_schwarzschild_vacuum = True
  M=5.0, r=10.0: weyl_squared_schwarzschild_vacuum = True
  M=0.5, r=1.5: weyl_squared_schwarzschild_vacuum = True

GB - C² = (2/3) R² - 2 Ricci² identity (random inputs):
  (R²= +8.12, Ric²=-30.52, Riem²=+46.53): identity holds = True
  (R²=+42.40, Ric²= -3.29, Riem²=+16.35): identity holds = True
  (R²=-28.55, Ric²=-27.83, Riem²=-21.15): identity holds = True
  (R²=+19.24, Ric²=-28.76, Riem²=+47.11): identity holds = True
  (R²=-42.96, Ric²=-30.67, Riem²=-41.11): identity holds = True


## 4. Stelle-basis coefficients (α, β, γ)

From the linear system

\begin{align*}
\alpha + \tfrac{\beta}{3} + \gamma &= c_R = -\tfrac{5}{2160},\\
-2\beta - 4\gamma &= c_{\rm Ric} = \tfrac{7}{2160},\\
\beta + \gamma &= c_{\rm Riem} = -\tfrac{1}{180},
\end{align*}

solving:

$$
\alpha = -\frac{N_f}{324\,(4\pi)^2},\quad
\beta  = -\frac{41\,N_f}{4320\,(4\pi)^2},\quad
\gamma = +\frac{17\,N_f}{4320\,(4\pi)^2}.
$$

**Lean defs:** `a4_alpha`, `a4_beta`, `a4_gamma`. Sign-definite for $N_f > 0$ via `a4_alpha_neg`, `a4_beta_neg`, `a4_gamma_pos`.

In [4]:
from src.higher_curvature import stelle_basis_coefficients

print(f"{'N_f':>6} {'α':>14} {'β':>14} {'γ':>14}   sign(α<0,β<0,γ>0)")
print("-" * 75)
for n in (1, 5, 24, 27, 100):
    c = stelle_basis_coefficients(float(n))
    sgn = (c.alpha < 0, c.beta < 0, c.gamma > 0)
    print(f"{n:>6}  {c.alpha:+12.4e}  {c.beta:+12.4e}  {c.gamma:+12.4e}   {sgn}")

   N_f              α              β              γ   sign(α<0,β<0,γ>0)
---------------------------------------------------------------------------
     1   -1.9545e-05   -6.0101e-05   +2.4920e-05   (True, True, True)
     5   -9.7725e-05   -3.0050e-04   +1.2460e-04   (True, True, True)
    24   -4.6908e-04   -1.4424e-03   +5.9808e-04   (True, True, True)
    27   -5.2771e-04   -1.6227e-03   +6.7284e-04   (True, True, True)
   100   -1.9545e-03   -6.0101e-03   +2.4920e-03   (True, True, True)


## 5. Main basis-change identity (Lean cross-bridge)

**Lean theorem `a4_density_eq_a4_density_in_RC2GB_basis`:**
For all $N_f, R^2, R_{\mu\nu}^2, R_{\mu\nu\rho\sigma}^2 \in \mathbb{R}$,

$$
c_R(N_f) R^2 + c_{\rm Ric}(N_f) R_{\mu\nu}^2 + c_{\rm Riem}(N_f) R_{\mu\nu\rho\sigma}^2
=
\alpha(N_f) R^2 + \beta(N_f) C^2 + \gamma(N_f) \mathcal{G}.
$$

Proof: `unfold a4_density a4_density_in_RC2GB_basis a4_R_sq_coef a4_Ricci_sq_coef a4_Riemann_sq_coef a4_alpha a4_beta a4_gamma weylSquared4D gaussBonnet4D; ring`. Substantive cross-bridge — the proof body invokes Wave 1 names by reference (P6 drift-protection).

In [5]:
from src.higher_curvature import a4_density, a4_density_in_RC2GB_basis

print("Random-input residuals (should be O(machine epsilon)):")
import random
random.seed(2026_04_30)
max_resid = 0.0
for _ in range(10):
    N_f = random.uniform(1.0, 100.0)
    R, Ric, Riem = (random.uniform(-100, 100) for _ in range(3))
    a = a4_density(N_f, R, Ric, Riem)
    b = a4_density_in_RC2GB_basis(N_f, R, Ric, Riem)
    resid = abs(a - b)
    max_resid = max(max_resid, resid)
    print(f"  N_f={N_f:6.2f}  resid={resid:.2e}")
print(f"\nmax residual = {max_resid:.2e}  (machine epsilon)")

Random-input residuals (should be O(machine epsilon)):
  N_f= 32.15  resid=2.78e-17
  N_f= 27.66  resid=6.94e-18
  N_f= 86.77  resid=1.11e-16
  N_f= 97.67  resid=1.67e-16
  N_f= 65.67  resid=1.11e-16
  N_f= 53.12  resid=2.78e-17
  N_f= 35.56  resid=5.55e-17
  N_f= 24.27  resid=4.16e-17
  N_f= 21.59  resid=4.16e-17
  N_f= 22.07  resid=4.16e-17

max residual = 1.67e-16  (machine epsilon)


## 6. Observational ceilings on dimensionless higher-curvature couplings

From Calmet, Capozziello & Pryer (arXiv:1905.13728) and Berti et al (LRR 18, 1, 2015):

| Source | Coupling | Ceiling |
|---|---|---|
| LIGO/Virgo (GW170817) | $|\beta|$ ($C^2$) | $\sim 10^{62}$ |
| Eöt-Wash 50 μm | $|\alpha|$ ($R^2$) | $\sim 10^{61}$ |
| Hulse-Taylor binary pulsar | $|\beta|$ ($C^2$) | $\sim 10^{59}$  ← tightest |
| Cassini PN solar-system | $|\beta|$ ($C^2$) | $\sim 10^{62}$ |

**Lean defs:** `hc_bound_LIGO`, `hc_bound_SRG`, `hc_bound_pulsar`, `hc_bound_cassini`.

In [6]:
from src.higher_curvature import HC_OBS_BOUNDS

for k, v in HC_OBS_BOUNDS.items():
    print(f"  {k:<16} = {v:.2e}")
print(f"\nPulsar tightest? {HC_OBS_BOUNDS['pulsar_C_sq'] < HC_OBS_BOUNDS['LIGO_C_sq']}")

  LIGO_C_sq        = 1.00e+62
  SRG_R_sq         = 1.00e+61
  pulsar_C_sq      = 1.00e+59
  cassini_C_sq     = 1.00e+62

Pulsar tightest? True


## 7. Correctness-push: predictions vs ceilings

**Lean theorem `higher_curvature_below_pulsar_bound`:** for $0 < N_f \le 100$, every Wave 1 a_4 coefficient is strictly below `hc_bound_pulsar = 10⁵⁹`. The 3-conjunct bundle is *not* P2-redundant — each conjunct invokes a distinct Wave 1 coefficient and a distinct positivity proof.

Proof structure: $1 < (4\pi)^2$ (from `Real.pi_gt_three` + `nlinarith`) → $(4\pi)^{-2} < 1$ → $100 \cdot 12/2160 \cdot 1 < 1$ → $1 < 10^{59}$.

In [7]:
from src.higher_curvature import largest_predicted_coefficient, predictions_below_bound, pulsar_correctness_push_passes

for n in (24, 27, 100):
    largest = largest_predicted_coefficient(float(n))
    passes = predictions_below_bound(float(n), HC_OBS_BOUNDS['pulsar_C_sq'])
    print(f"N_f={n}: largest_predicted = {largest:.4e}, below_pulsar_bound = {passes}")

print()
print(f"Wave 2 correctness-push (default N_f=24): {pulsar_correctness_push_passes()}")
print(f"Wave 2 correctness-push (SM+ν_R, N_f=27):  {pulsar_correctness_push_passes(27)}")
print(f"Wave 2 correctness-push (high N_f=100):   {pulsar_correctness_push_passes(100)}")

N_f=24: largest_predicted = 8.4434e-04, below_pulsar_bound = True
N_f=27: largest_predicted = 9.4989e-04, below_pulsar_bound = True
N_f=100: largest_predicted = 3.5181e-03, below_pulsar_bound = True

Wave 2 correctness-push (default N_f=24): True
Wave 2 correctness-push (SM+ν_R, N_f=27):  True
Wave 2 correctness-push (high N_f=100):   True


## 8. Falsifier: predictions strictly non-zero

**Lean theorem `higher_curvature_predictions_strictly_positive`:** for $N_f > 0$, all three Wave 1 a_4 coefficient absolute values are strictly positive.

This rules out the trivial reading "all bounds are passed because all predictions are zero" — the bounds are passed because the predictions are tiny but non-trivial.

In [8]:
print("Falsifier check (artificial sub-prediction bound):")
for n in (24, 27, 100):
    fails = not predictions_below_bound(float(n), 1e-10)
    print(f"  N_f={n}: predictions NOT below 1e-10 = {fails}  (expected True)")

Falsifier check (artificial sub-prediction bound):
  N_f=24: predictions NOT below 1e-10 = True  (expected True)
  N_f=27: predictions NOT below 1e-10 = True  (expected True)
  N_f=100: predictions NOT below 1e-10 = True  (expected True)


## 9. Tracked Prop H_HigherCurvatureWithinObservationalBounds

**Lean def `H_HigherCurvatureWithinObservationalBounds (B : ℝ) : Prop`** parameterised by upper bound $B$:

$$
H_{\rm HC}(B) :\; 0 < B \;\wedge\; \forall N_f,\; 0 < N_f \le 100,\; |c_R| \le B \wedge |c_{\rm Ric}| \le B \wedge |c_{\rm Riem}| \le B.
$$

**Lean witness theorem `H_HigherCurvatureWithinObservationalBounds_pulsar_witness`:** $H_{\rm HC}({\rm pulsar} = 10^{59})$ holds, proved from the correctness-push.

## 10. Figure: predictions vs ceilings

In [9]:
# viz-ref: fig_higher_curvature_obs_bounds
from src.core.visualizations import fig_higher_curvature_obs_bounds
fig = fig_higher_curvature_obs_bounds()
fig.show()

## 11. Lean theorem inventory

All 11 substantive theorems shipped in `lean/SKEFTHawking/HigherCurvatureStructure.lean`:

| # | Name | Role |
|---|---|---|
| 1 | `weylSquared4D_eq_zero_iff_conformally_flat` | conformal-flatness biconditional |
| 2 | `gaussBonnet_minus_weyl_eq_R_minus_Ricci_combination` | algebraic engine |
| 3 | `a4_alpha_neg` | $\alpha < 0$ for $N_f > 0$ |
| 4 | `a4_beta_neg` | $\beta < 0$ for $N_f > 0$ |
| 5 | `a4_gamma_pos` | $\gamma > 0$ for $N_f > 0$ (chiral-anomaly sign) |
| 6 | `a4_density_eq_a4_density_in_RC2GB_basis` | **MAIN basis-change identity** |
| 7 | `fourPiSq_gt_one` | helper: $1 < (4\pi)^2$ |
| 8 | `fourPiSqInv_lt_one` | helper: $(4\pi)^{-2} < 1$ |
| 9 | `higher_curvature_below_pulsar_bound` | **CORRECTNESS-PUSH** (3-conjunct) |
| 10 | `higher_curvature_predictions_strictly_positive` | **FALSIFIER** (3-conjunct) |
| 11 | `H_HigherCurvatureWithinObservationalBounds_pulsar_witness` | tracked-Prop instance |

Verified via `lean_verify`: axioms = `propext`, `Classical.choice`, `Quot.sound` only (Lean 4 kernel standard); zero new axioms.